# 04. Periodo, energía y potencia de una señal

**Objetivos de aprendizaje.** Al terminar podrás:

1. Calcular la **energía** y la **potencia** de una señal discreta (a mano, con un ciclo y con numpy).
2. Clasificar una señal como **de energía** o **de potencia**.
3. Relacionar potencia, valor **RMS** y amplitud, por ejemplo en los 127 V de la instalación de tu casa.
4. Encontrar el **periodo** de una señal: teóricamente (con $f = k/N$) y a partir de los datos.
5. Calcular el periodo de una **suma** de senoidales.

**Requisito:** notebooks 01 y 02.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. Energía

$$E = \sum_{n=-\infty}^{\infty} |x[n]|^2$$

**¿Por qué "energía"?** Si $x[n]$ fuera el voltaje en una resistencia de 1 Ω, $|x|^2$ sería la potencia instantánea,
y sumarla en el tiempo da energía. En PDS se usa el mismo nombre aunque la señal no sea un voltaje.

### Paso a paso con $x[n] = \{1, -2, 3\}$

In [ ]:
x = np.array([1.0, -2.0, 3.0])

E = 0.0                                   # acumulador
for i in range(len(x)):
    cuadrado = abs(x[i]) ** 2
    E = E + cuadrado
    print(f"  muestra {i}: |{x[i]:4.1f}|² = {cuadrado:4.1f}   ->  E parcial = {E}")

print("E =", E)

In [ ]:
# La misma cuenta con la biblioteca y con numpy en una línea
print("pds.energia(x)       =", pds.energia(x))
print("np.sum(np.abs(x)**2) =", np.sum(np.abs(x) ** 2))

## 2. Potencia promedio

$$P = \lim_{N\to\infty} \frac{1}{2N+1}\sum_{n=-N}^{N} |x[n]|^2$$

En la práctica sólo tenemos un número finito de muestras, así que calculamos el **promedio de $|x[n]|^2$**:
$P = E/N$.

### Señales de energía vs señales de potencia

| Tipo | Energía | Potencia | Ejemplos |
|---|---|---|---|
| de **energía** | finita ($0 < E < \infty$) | $P = 0$ | impulso, pulso, $0.9^n u[n]$ |
| de **potencia** | infinita | finita ($0 < P < \infty$) | escalón, senoidal, señales periódicas |
| ninguna | infinita | infinita | $2^n u[n]$, $n\,u[n]$ |

Comprobémoslo numéricamente: calculamos $E$ y $P$ en ventanas cada vez más grandes $[-N, N]$.
Si $E$ se estabiliza → señal de energía. Si $P$ se estabiliza (y $E$ crece) → señal de potencia.

In [ ]:
print(f"{'N':>6} | {'E pulso':>8} {'P pulso':>9} | {'E escalón':>10} {'P escalón':>10}")
print("-" * 54)
for N in [10, 100, 1000, 10000]:
    n, pulso = pds.escalon(-N, N, n0=0)
    n, u5 = pds.escalon(-N, N, n0=5)
    pulso = pulso - u5                       # pulso de 5 muestras
    n, escalon = pds.escalon(-N, N)

    print(f"{N:>6} | {pds.energia(pulso):8.1f} {pds.e_promedio(pulso):9.5f} | "
          f"{pds.energia(escalon):10.1f} {pds.e_promedio(escalon):10.5f}")

- El **pulso**: $E = 5$ siempre, $P \to 0$ → señal de **energía**.
- El **escalón**: $E$ crece sin límite, $P \to 1/2$ (la mitad del tiempo vale 1) → señal de **potencia**.

## 3. Potencia de una senoidal y valor RMS

Para una senoidal de amplitud $A$:
$$P = \frac{A^2}{2} \qquad\qquad x_{\text{rms}} = \sqrt{P} = \frac{A}{\sqrt 2}$$

Como es periódica, basta con promediar sobre **un periodo** (`pds.potencia_periodica`).

In [ ]:
A = 3
n, x = pds.senoidal_digital_muestras(N=100, f=1/20, A=A)

P_un_periodo = pds.potencia_periodica(x, N=20)
print("Potencia (1 periodo)   =", P_un_periodo)
print("Teórica A²/2           =", A**2 / 2)
print("Valor RMS              =", pds.valor_rms(x))
print("Teórico A/√2           =", A / np.sqrt(2))

> **Conexión con ingeniería eléctrica:** cuando decimos que la red es de "127 V", nos referimos al valor **RMS**.
> La amplitud (el pico) es $127\sqrt2 \approx 180$ V. El valor RMS es el voltaje de CD que disiparía la **misma potencia**
> en una resistencia.

## 4. Periodo de una señal

### 4.1 Teórico: senoidal digital
Ya lo vimos en el notebook 01: si $f = k/N$ (fracción irreducible) el periodo es $N$.

### 4.2 Suma de señales periódicas
Si $x_1$ tiene periodo $N_1$ y $x_2$ periodo $N_2$, la suma se repite cuando **ambas** se repiten a la vez:
$$N = \text{mcm}(N_1, N_2)$$

In [ ]:
import math

N1 = pds.periodo_fundamental_digital(1/4)
N2 = pds.periodo_fundamental_digital(1/6)
N_suma = math.lcm(N1, N2)
print(f"N1 = {N1}, N2 = {N2}  ->  N = mcm(N1, N2) = {N_suma}")

n = np.arange(0, 48)
x = np.cos(2*np.pi*n/4) + 0.5*np.sin(2*np.pi*n/6)
pds.graficar_discreta(n, x, titulo=f"cos(2πn/4) + 0.5 sin(2πn/6): periodo {N_suma}")
for k in range(0, 48, N_suma):
    plt.axvline(k, color="r", linestyle="--", alpha=0.5)
plt.show()

### 4.3 A partir de los datos: búsqueda directa

Si sólo tenemos las muestras, probamos $N = 1, 2, 3, \dots$ y nos quedamos con el **primero** que cumpla
$x[n+N] = x[n]$ para todas las $n$ (con una pequeña tolerancia por el redondeo de la computadora).

In [ ]:
import inspect
print(inspect.getsource(pds.es_periodica))
print(inspect.getsource(pds.estimar_periodo))

In [ ]:
print("Periodo estimado:", pds.estimar_periodo(x))

> Ojo: este método **falla con señales reales**: con ruido, $x[n+N]$ nunca es *exactamente* $x[n]$.
> En el notebook 07 usaremos la **autocorrelación**, que funciona aun con mucho ruido.

## Resumen

| Concepto | Fórmula | Función |
|---|---|---|
| Energía | $E = \sum \lvert x[n]\rvert^2$ | `pds.energia(x)` |
| Potencia promedio | $P = \frac{1}{N}\sum \lvert x[n]\rvert^2$ | `pds.e_promedio(x)` |
| Potencia periódica | $P = \frac{1}{N}\sum_{n=0}^{N-1} \lvert x[n]\rvert^2$ | `pds.potencia_periodica(x, N)` |
| RMS | $\sqrt{P}$ | `pds.valor_rms(x)` |
| Periodo (teórico) | $f = k/N$ | `pds.periodo_fundamental_digital(f)` |
| Periodo (datos) | menor $N$ con $x[n+N]=x[n]$ | `pds.estimar_periodo(x)` |

## Comprueba tu comprensión

1. ¿Cuál es la energía de $\delta[n-5]$? ¿Y de $3\delta[n]$?
2. ¿$\cos(\pi n/3)$ es de energía o de potencia? ¿Cuánto vale su potencia?
3. ¿Cuál es el periodo de $\cos(2\pi n/6) + \cos(2\pi n/9)$?

<details><summary>Ver respuestas</summary>

1. $E = 1$; $E = 9$.
2. De potencia; $P = 1/2$.
3. $\text{mcm}(6, 9) = 18$.
</details>